In [ ]:
import os, subprocess
_nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
_req = f"/Workspace{os.path.dirname(_nb)}/requirements-serving.txt"
subprocess.check_call(["pip", "install", "-q", "-r", _req])
dbutils.library.restartPython()

# Ask Jorge — Serving Model

Builds, logs, registers, and deploys the RAG chain that powers the `jorge_cv_endpoint` serving endpoint.

After endpoint creation/update, the notebook configures **AI Gateway** via `put_ai_gateway` — idempotent on every re-deploy.

**AI Gateway features enabled**
| Feature | Where it logs |
|---|---|
| Usage tracking | `system.serving.endpoint_usage` — token counts, latency, requester |
| Inference tables | `jorge.cv_rag.jorge_cv_endpoint_payload` — full request + response JSON |

**Stack**
- LangChain RAG chain (retriever + ChatDatabricks + StrOutputParser)
- MLflow 3 logging + Unity Catalog model registration
- Databricks Vector Search (`jorge_cv_search` endpoint)
- Databricks SDK `put_ai_gateway`

In [ ]:
# ── Project constants ──────────────────────────────────────────────────────────
CATALOG           = "jorge"
DB                = "cv_rag"
MODEL_NAME        = "jorge_cv_rag_chain"
MODEL_NAME_FQN    = f"{CATALOG}.{DB}.{MODEL_NAME}"
ENDPOINT_NAME     = "jorge_cv_endpoint"

VS_ENDPOINT_NAME  = "jorge_cv_search"
VS_INDEX_NAME     = f"{CATALOG}.{DB}.jorge_cv_search_index"

# Foundation model used by the chain
LLM_ENDPOINT      = "databricks-meta-llama-3-3-70b-instruct"

chain_config = {
    "llm_endpoint_name":          LLM_ENDPOINT,
    "vector_search_endpoint_name": VS_ENDPOINT_NAME,
    "vector_search_index":         VS_INDEX_NAME,
    "llm_prompt_template": (
        "You are Jorge Martínez Zapico — a Senior MLOps & AI Engineer "
        "with expertise in Databricks, Azure, LangChain, and RAG systems. "
        "Answer questions about your professional background, projects, and skills "
        "in first person, using only the context provided. "
        "If the context does not contain enough information, say so honestly.\n\n"
        "Context:\n{context}"
    ),
}

print("Config ready:", chain_config)

In [ ]:
import mlflow
from databricks.vector_search.client import VectorSearchClient
from databricks_langchain.vectorstores import DatabricksVectorSearch
from databricks_langchain.chat_models import ChatDatabricks
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from operator import itemgetter

mlflow.langchain.autolog()

model_config = mlflow.models.ModelConfig(development_config=chain_config)

# ── Retriever ──────────────────────────────────────────────────────────────────
vector_search_as_retriever = DatabricksVectorSearch(
    endpoint=model_config.get("vector_search_endpoint_name"),
    index_name=model_config.get("vector_search_index"),
    columns=["id", "chunk_text"],
).as_retriever(search_kwargs={"k": 5})

mlflow.models.set_retriever_schema(primary_key="id", text_column="chunk_text")


def format_context(docs):
    return "".join(f"Passage: {d.page_content}\n" for d in docs)


# ── Prompt + LLM ──────────────────────────────────────────────────────────────
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", model_config.get("llm_prompt_template")),
        ("user", "{question}"),
    ]
)

llm = ChatDatabricks(
    endpoint=model_config.get("llm_endpoint_name"),
    extra_params={"temperature": 0.01, "max_tokens": 500},
)


def extract_user_query(messages):
    return messages[-1]["content"]


# ── RAG chain ─────────────────────────────────────────────────────────────────
chain = (
    {
        "question": itemgetter("messages") | RunnableLambda(extract_user_query),
        "context":  itemgetter("messages")
                    | RunnableLambda(extract_user_query)
                    | vector_search_as_retriever
                    | RunnableLambda(format_context),
    }
    | prompt
    | llm
    | StrOutputParser()
)

# Quick smoke test
test_input = {"messages": [{"role": "user", "content": "What is Jorge's current role?"}]}
answer = chain.invoke(test_input)
print("Smoke test answer:", answer)

In [ ]:
# ── Write chain.py so MLflow can serialize it ─────────────────────────────────
chain_py = '''
from databricks.vector_search.client import VectorSearchClient
from databricks_langchain.vectorstores import DatabricksVectorSearch
from databricks_langchain.chat_models import ChatDatabricks
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from operator import itemgetter
import mlflow

mlflow.langchain.autolog()

model_config = mlflow.models.ModelConfig(development_config="chain_config.yaml")

vector_search_as_retriever = DatabricksVectorSearch(
    endpoint=model_config.get("vector_search_endpoint_name"),
    index_name=model_config.get("vector_search_index"),
    columns=["id", "chunk_text"],
).as_retriever(search_kwargs={"k": 5})

mlflow.models.set_retriever_schema(primary_key="id", text_column="chunk_text")

def format_context(docs):
    return "".join(f"Passage: {d.page_content}\\n" for d in docs)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", model_config.get("llm_prompt_template")),
        ("user", "{question}"),
    ]
)

llm = ChatDatabricks(
    endpoint=model_config.get("llm_endpoint_name"),
    extra_params={"temperature": 0.01, "max_tokens": 500},
)

def extract_user_query(messages):
    return messages[-1]["content"]

chain = (
    {
        "question": itemgetter("messages") | RunnableLambda(extract_user_query),
        "context":  itemgetter("messages")
                    | RunnableLambda(extract_user_query)
                    | vector_search_as_retriever
                    | RunnableLambda(format_context),
    }
    | prompt
    | llm
    | StrOutputParser()
)

mlflow.models.set_model(model=chain)
'''

import yaml

with open("chain.py", "w") as f:
    f.write(chain_py)

with open("chain_config.yaml", "w") as f:
    yaml.dump(chain_config, f)

print("chain.py and chain_config.yaml written.")

In [ ]:
from mlflow.models.resources import DatabricksVectorSearchIndex, DatabricksServingEndpoint
from mlflow.types.llm import ChatCompletionRequest

input_example = {"messages": [{"role": "user", "content": "What is Jorge's current role?"}]}

mlflow.set_experiment(f"/Shared/{MODEL_NAME}")

with mlflow.start_run(run_name="jorge_cv_rag_deploy"):
    logged_chain_info = mlflow.langchain.log_model(
        lc_model=os.path.join(os.getcwd(), "chain.py"),
        name="chain",
        model_config=chain_config,
        input_example=input_example,
        resources=[
            DatabricksVectorSearchIndex(index_name=VS_INDEX_NAME),
            DatabricksServingEndpoint(endpoint_name=LLM_ENDPOINT),
        ],
    )

print("Model URI:", logged_chain_info.model_uri)

# Register to Unity Catalog
uc_model = mlflow.register_model(
    model_uri=logged_chain_info.model_uri,
    name=MODEL_NAME_FQN,
)
print(f"Registered: {MODEL_NAME_FQN} v{uc_model.version}")

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput,
    AiGatewayUsageTrackingConfig,
    AiGatewayInferenceTableConfig,
)

w = WorkspaceClient()

served_entities = [
    ServedEntityInput(
        entity_name=MODEL_NAME_FQN,
        entity_version=uc_model.version,
        workload_size="Small",
        scale_to_zero_enabled=True,
    )
]

# ── Create or update the endpoint ─────────────────────────────────────────────
try:
    existing = w.serving_endpoints.get(ENDPOINT_NAME)
    print(f"Endpoint '{ENDPOINT_NAME}' exists — updating config...")
    w.serving_endpoints.update_config_and_wait(
        name=ENDPOINT_NAME,
        served_entities=served_entities,
    )
    print("Endpoint updated and ready.")
except Exception:
    print(f"Endpoint '{ENDPOINT_NAME}' not found — creating...")
    w.serving_endpoints.create_and_wait(
        name=ENDPOINT_NAME,
        config=EndpointCoreConfigInput(
            name=ENDPOINT_NAME,
            served_entities=served_entities,
        ),
    )
    print(f"Endpoint '{ENDPOINT_NAME}' created and ready.")

In [ ]:
# ── AI Gateway: usage tracking + inference tables (payload logging) ────────────
#
# usage_tracking_config  → logs token counts, latency, requester to:
#                           system.serving.endpoint_usage
#                           system.serving.served_entities
#
# inference_table_config → logs the FULL request + response JSON to:
#                           jorge.cv_rag.jorge_cv_endpoint_payload
#                          (available ~1 h after each request)
#
# Both calls are idempotent — safe to re-run after every re-deploy.
# API ref: PUT /api/2.0/serving-endpoints/{name}/ai-gateway

gateway_response = w.serving_endpoints.put_ai_gateway(
    name=ENDPOINT_NAME,
    usage_tracking_config=AiGatewayUsageTrackingConfig(enabled=True),
    inference_table_config=AiGatewayInferenceTableConfig(
        enabled=True,
        catalog_name=CATALOG,
        schema_name=DB,
        table_name_prefix=ENDPOINT_NAME,  # table → jorge.cv_rag.jorge_cv_endpoint_payload
    ),
)

tracking_enabled = (
    gateway_response.usage_tracking_config is not None
    and gateway_response.usage_tracking_config.enabled
)
inference_enabled = (
    gateway_response.inference_table_config is not None
    and gateway_response.inference_table_config.enabled
)

print(f"AI Gateway usage tracking : {tracking_enabled}")
print(f"AI Gateway inference table: {inference_enabled}")
print(f"\nPayload table: {CATALOG}.{DB}.{ENDPOINT_NAME}_payload")
print(f"  → stores the full question + answer JSON for every request")
print(f"  → query with: SELECT * FROM {CATALOG}.{DB}.{ENDPOINT_NAME}_payload")
print(f"\nUsage table  : system.serving.endpoint_usage")
print(f"  → token counts, latency, requester (account admin access)")
print(f"\nDone. Endpoint: {ENDPOINT_NAME} | Model: {MODEL_NAME_FQN} v{uc_model.version}")